In [88]:
import numpy as np
import pandas as pd

np.random.seed(42)

# =========================================================
# 1. BASIC SETTINGS
# =========================================================

n = 200000

# =========================================================
# 2. COLLEGE TIER
# =========================================================

college_tier = np.random.choice(
    ["Tier 1", "Tier 2", "Tier 3"],
    size=n,
    p=[0.15, 0.35, 0.50]
)

# =========================================================
# 3. GENDER
# =========================================================

gender = np.random.choice(
    ["Male", "Female"],
    size=n,
    p=[0.55, 0.45]
)

# =========================================================
# 4. HOURS STUDYING COLLEGE SYLLABUS
# =========================================================

hours_college = np.random.gamma(
    shape=2.5,
    scale=2.0,
    size=n
)

hours_college = np.clip(hours_college, 0.5, 15)

# =========================================================
# 5. HOURS STUDYING SKILLS
# =========================================================

hours_skills = np.random.gamma(
    shape=2.2,
    scale=2.0,
    size=n
)

hours_skills = np.clip(hours_skills, 0.5, 15)

# =========================================================
# 6. MHT-CET PERCENTILE
# =========================================================

mhtcet = np.random.normal(
    loc=82,
    scale=12,
    size=n
)

mhtcet = np.clip(mhtcet, 40, 99.99)

# =========================================================
# 7. JEE PERCENTILE
# =========================================================

jee = (
    mhtcet * 0.65
    + np.random.normal(30, 12, n)
)

jee = np.clip(jee, 35, 99.99)

# =========================================================
# 8. COLLEGE TIER EFFECT
# =========================================================

tier_effect = np.select(
    [
        college_tier == "Tier 1",
        college_tier == "Tier 2",
        college_tier == "Tier 3"
    ],
    [
        8.0,
        4.0,
        0.0
    ]
)

# =========================================================
# 9. ACADEMIC EFFECT
# =========================================================

academic_effect = (
    0.045 * mhtcet
    + 0.025 * jee
)

# =========================================================
# 10. COLLEGE STUDY EFFECT
# =========================================================

# Diminishing returns
college_study_effect = (
    1.1 * np.sqrt(hours_college)
)

# =========================================================
# 11. SKILL DEVELOPMENT EFFECT
# =========================================================

skill_effect = (
    1.5 * hours_skills
    + 0.10 * hours_skills ** 2
)

# =========================================================
# 12. INTERACTION:
# HIGH SKILLS + BETTER COLLEGE
# =========================================================

interaction = np.where(
    college_tier == "Tier 1",
    0.35 * hours_skills,
    np.where(
        college_tier == "Tier 2",
        0.15 * hours_skills,
        0
    )
)

# =========================================================
# 13. ACADEMICS × SKILLS INTERACTION
# =========================================================

academic_skill_interaction = (
    0.0008
    * mhtcet
    * hours_skills
)

# =========================================================
# 14. GENDER
# =========================================================

# Almost no systematic effect.
# Mainly noise rather than a meaningful predictor.

gender_effect = np.where(
    gender == "Female",
    0.15,
    0
)

# =========================================================
# 15. RANDOM MARKET / COMPANY NOISE
# =========================================================

noise = np.random.normal(
    loc=0,
    scale=3.5,
    size=n
)

# =========================================================
# 16. SOME OUTLIERS
# =========================================================

outlier_mask = np.random.random(n) < 0.025

noise[outlier_mask] += np.random.normal(
    0,
    8,
    outlier_mask.sum()
)

# =========================================================
# 17. FINAL PACKAGE
# =========================================================

package = (
    2
    + tier_effect
    + academic_effect
    + college_study_effect
    + skill_effect
    + interaction
    + academic_skill_interaction
    + gender_effect
    + noise
)

# Keep packages realistic
package = np.clip(package, 2.5, 45)

# =========================================================
# 18. CREATE DATAFRAME
# =========================================================

df = pd.DataFrame({
    "Hours_College_Syllabus": np.round(hours_college, 2),
    "Hours_Skill_Development": np.round(hours_skills, 2),
    "College_Tier": college_tier,
    "MHTCET_Percentile": np.round(mhtcet, 2),
    "JEE_Percentile": np.round(jee, 2),
    "Gender": gender,
    "Final_Package_LPA": np.round(package, 2)
})

# =========================================================
# 19. INTRODUCE REALISTIC MISSING VALUES
# =========================================================

rng = np.random.default_rng(42)

# Different columns have different missing rates

missing_rates = {
    "Hours_College_Syllabus": 0.03,
    "Hours_Skill_Development": 0.04,
    "College_Tier": 0.02,
    "MHTCET_Percentile": 0.05,
    "JEE_Percentile": 0.07,
    "Gender": 0.02
}

for column, rate in missing_rates.items():

    mask = rng.random(n) < rate
    df.loc[mask, column] = np.nan

# =========================================================
# 20. SHUFFLE DATA
# =========================================================

df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# =========================================================
# 21. SAVE
# =========================================================

df.to_csv(
    "student_package_dataset.csv",
    index=False
)

df.head(10)
# print("\nShape:", df.shape)
# print("\nMissing values:")
# print(df.isna().sum())

# print("\nStatistics:")
# print(df.describe(include="all"))

,Hours_College_Syllabus,Hours_Skill_Development,College_Tier,MHTCET_Percentile,JEE_Percentile,Gender,Final_Package_LPA
0,1.57,4.93,Tier 3,78.51,NaN,Male,16.57
1,6.93,3.80,NaN,82.06,79.81,Female,15.26
2,3.34,5.36,Tier 2,71.88,77.49,Female,35.26
3,NaN,2.45,Tier 3,75.86,73.64,Male,13.89
4,1.92,2.73,Tier 2,89.24,99.99,Male,17.64
5,2.52,3.16,Tier 2,NaN,95.67,Male,19.72
6,3.66,3.65,Tier 2,85.25,89.00,Male,20.76
7,9.45,3.41,Tier 2,75.28,77.96,Male,20.43
8,7.43,6.02,Tier 3,91.27,99.99,Male,22.20
9,1.83,8.61,Tier 3,99.99,80.58,Female,28.23


In [89]:
df.describe()

,Hours_College_Syllabus,Hours_Skill_Development,MHTCET_Percentile,JEE_Percentile,Final_Package_LPA
count,194005.000000,192076.000000,189987.000000,186054.000000,200000.000000
mean,4.975888,4.377859,81.645446,82.273502,22.510143
std,3.065636,2.898637,11.342969,12.784735,9.014873
min,0.500000,0.500000,40.000000,35.000000,2.500000
25%,2.660000,2.210000,73.890000,73.550000,16.040000
50%,4.360000,3.740000,82.010000,83.170000,21.090000
75%,6.630000,5.870000,90.115000,92.600000,27.460000
max,15.000000,15.000000,99.990000,99.990000,45.000000


In [90]:
df.head()

,Hours_College_Syllabus,Hours_Skill_Development,College_Tier,MHTCET_Percentile,JEE_Percentile,Gender,Final_Package_LPA
0,1.57,4.93,Tier 3,78.51,NaN,Male,16.57
1,6.93,3.80,NaN,82.06,79.81,Female,15.26
2,3.34,5.36,Tier 2,71.88,77.49,Female,35.26
3,NaN,2.45,Tier 3,75.86,73.64,Male,13.89
4,1.92,2.73,Tier 2,89.24,99.99,Male,17.64


In [91]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import chi2, SelectKBest

In [92]:
df.head(1)

,Hours_College_Syllabus,Hours_Skill_Development,College_Tier,MHTCET_Percentile,JEE_Percentile,Gender,Final_Package_LPA
0,1.57,4.93,Tier 3,78.51,NaN,Male,16.57


In [93]:
numeric_cols = ["Hours_College_Syllabus", "Hours_Skill_Development", "MHTCET_Percentile","JEE_Percentile"]
categorical_cols_Ordinal = ["College_Tier"]
categorical_cols_Nominal = ["Gender"]

In [94]:
# Numerical Data pipeline
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('Scalar', MinMaxScaler())
])

categorical_Nominal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(sparse_output=False, drop='first'))
])

categorical_Ordinal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('Nominal', OrdinalEncoder(categories=[['Tier 3', 'Tier 2', 'Tier 1']]))
])

In [95]:
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipe, numeric_cols),
    ('Ncat', categorical_Nominal_pipe, categorical_cols_Nominal),
    ('Ocat', categorical_Ordinal_pipe, categorical_cols_Ordinal)
], remainder='passthrough')

In [96]:
pipe = Pipeline([
    ('Preprocessing', preprocessor),
    ('model', LinearRegression())
])

In [97]:
df.sample(5)

,Hours_College_Syllabus,Hours_Skill_Development,College_Tier,MHTCET_Percentile,JEE_Percentile,Gender,Final_Package_LPA
77811,6.84,4.03,Tier 2,85.04,74.33,Female,20.26
63094,1.86,1.03,Tier 3,74.50,NaN,Male,13.81
97798,4.13,1.19,Tier 1,91.28,NaN,NaN,18.63
56904,0.50,5.95,Tier 1,71.02,74.12,Female,29.58
29656,3.44,2.63,Tier 2,93.64,75.54,Male,18.44


In [98]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=["Final_Package_LPA"]), df["Final_Package_LPA"], test_size= 0.25)

In [99]:
pipe.fit(X_test, y_test)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['Hours_College_Syllabus','Hours_Skill_Development','College_Tier', 'MHTCET_Percentile','JEE_Percentile','Gender']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('Ncat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be 

In [100]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

y_train_pred = pipe.predict(X_train)
y_pred = pipe.predict(X_test)
print("TRAIN")
print("MAE :", mean_absolute_error(y_train, y_train_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_train, y_train_pred)))
print("R²  :", r2_score(y_train, y_train_pred))

print("\nTEST")
print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²  :", r2_score(y_test, y_pred))

TRAIN
MAE : 3.115164166738211
RMSE: 4.085059098788068
R²  : 0.7951044506334759

TEST
MAE : 3.1134066959510167
RMSE: 4.082749668116136
R²  : 0.7935361194441524


In [101]:
import joblib
joblib.dump(pipe, "Students_Trained_Model.pkl")

['Students_Trained_Model.pkl']